<a href="https://colab.research.google.com/github/hamed85/ML-challenges/blob/main/CNN_TrafficSigneDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### This file is the main function for applying traffic sign recognition using LeNet-5 ConvNet.
### This project trains a model to decode traffic signs from natural images by using the German
### Traffic Sign Dataset. After training the model, it can be tested on new images of traffic signs.

# Import the required libraries into the main function:
#import pandas as pd
import numpy as np
import tensorflow as tf
#import pickle
#import matplotlib.pyplot as plt
#from IPython.display import Image
#from matplotlib import gridspec
import os
import numpy as np
from random import shuffle
#from PIL import Image
import tflearn
from tflearn.data_utils import image_preloader
from tflearn.data_utils import samplewise_zero_center
from tflearn.data_utils import samplewise_std_normalization
from tflearn.data_preprocessing import ImagePreprocessing
from tflearn.data_augmentation import ImageAugmentation
from sklearn.model_selection import train_test_split

one_passenger_dir   = 'datasets/one_passenger'
two_passenger_dir   = 'datasets/two_passenger'
three_passenger_dir = 'datasets/three_passenger'
four_passenger_dir  = 'datasets/four_passenger'

train_data_file     = 'datasets/train.txt'
test_data_file      = 'datasets/test.txt'
valid_data_file     = 'datasets/valid.txt'

# Define design parameters:
num_classes      = 4
learning_rate    = 0.001
reg_param        = 1e-4
channels         = 3        # We consdier 3 colour channels (RGB) for our images
img_size_height  = 360 #1080    # Height of images used for model training and validation
img_size_width   = 480 #1440    # Width of images used for model training and validation
train_proportion = 0.90     # Proportion of the data to be used for training
test_proportion  = 0.05     # Proportion of the data to be used for testing
valid_proportion = 0.05     # Proportion of the data to be used for model validation

def load_label_image(image_path):
    dataset = []
    for image in os.listdir(image_path):
        if image_path.split("/")[-1]   == 'one_passenger':
            label = 0
        elif image_path.split("/")[-1] == 'two_passenger':
            label = 1
        elif image_path.split("/")[-1] == 'three_passenger':
            label = 2
        elif image_path.split("/")[-1] == 'four_passenger':
            label = 3
        path  = os.path.join(image_path, image)
        dataset.append([path, label])
        shuffle(dataset)
    return dataset

# Let's now load and sample the dataset:
# Load [path, label] of the Cookpad Sandwish and Sushi images:
one_passenger_data   = load_label_image(one_passenger_dir)
two_passenger_data   = load_label_image(two_passenger_dir)
three_passenger_data = load_label_image(three_passenger_dir)
four_passenger_data  = load_label_image(four_passenger_dir)

# Merge both datasets into one full data:
full_dataset  = one_passenger_data + two_passenger_data + three_passenger_data + four_passenger_data

def split_image_dataset(dataset, test_proportion, valid_proportion):
    x = [row[0] for row in dataset]
    y = [row[1] for row in dataset]
    x_train, x_test, y_train, y_test   = train_test_split(x, y, test_size=test_proportion, stratify=y) #shuffle=True)
    x_train, x_valid, y_train, y_valid = train_test_split(x_train, y_train, test_size=valid_proportion, stratify=y_train)# shuffle=True)
    train_data = np.column_stack((x_train,y_train))
    valid_data = np.column_stack((x_valid,y_valid))
    test_data  = np.column_stack((x_test,y_test))
    return train_data, test_data, valid_data

# Now let's spilit the dataset into: Training, Test and Validation subsets:
# Startified sampling is used here to keep a balanced class distribution across the three subsets:
train_data, test_data, valid_data = split_image_dataset(full_dataset, test_proportion, valid_proportion)

# Save the the URL and Label of images in the three subsets
# Will use these files to load the images into the memory
np.savetxt(fname=train_data_file, X=train_data, delimiter=' ', fmt='%s')
np.savetxt(fname=test_data_file,  X=test_data,  delimiter=' ', fmt='%s')
np.savetxt(fname=valid_data_file, X=valid_data, delimiter=' ', fmt='%s')

# The next step is to load the images in the three image subsets. To this end, we will use <i>TFlearn Preloader</i> classes,
# which enable to handle large datasets, since one may be limited by memory (RAM) and it may not be possible to load the whole
# dataset at once. Thats when you want to use those classes and only load a part of the data on the fly. For example during
# the training phase only the batches will be loaded into the RAM when using a Preloader class.
# Load the images and their labels into the memroy:
train_x, train_y = image_preloader(train_data_file, image_shape=(img_size_height,img_size_width), categorical_labels=True, normalize=True)
test_x,  test_y  = image_preloader(test_data_file,  image_shape=(img_size_height,img_size_width), categorical_labels=True, normalize=True)
valid_x, valid_y = image_preloader(valid_data_file, image_shape=(img_size_height,img_size_width), categorical_labels=True, normalize=True)

train_x = samplewise_zero_center(train_x[:])
train_x = samplewise_std_normalization(train_x[:])
test_x  = samplewise_zero_center(test_x[:])
test_x  = samplewise_std_normalization(test_x[:])
valid_x = samplewise_zero_center(valid_x[:])
valid_x = samplewise_std_normalization(valid_x[:])

# Print some stats!
print("Number of training images:", format(len(train_x)))
print("Number of test images:", format(len(test_x)))
print("Number of validation images:", format(len(valid_x)))

### Step 2) Data Visualization:
##from visualization import show_images, show_distribution
##show_images(train_x, train_x, train_y, 5, 5)
##show_distribution(train_y)

# Step 3) Define the LeNet network structure:
from conv_LeNet import LeNet, reg_cost

# Define placeholders for input images, labels and dropout percentage
x           = tf.placeholder("float", [None, img_size_width, img_size_height, 3])
tf.add_to_collection("x", x)
y           = tf.placeholder("float", [None, num_classes])
tf.add_to_collection("y", y)
keep_prob   = tf.placeholder(tf.float32)
tf.add_to_collection("keep_prob", keep_prob)

# Define the LetNet network
logits, weights = LeNet(x, num_classes)
tf.add_to_collection("logits", logits)
loss_op     = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits=logits,labels=y)) + reg_cost(weights, reg_param)
tf.add_to_collection('loss_op', loss_op)

optimizer   = tf.train.AdamOptimizer(learning_rate).minimize(loss_op)
correct_prediction = tf.equal(tf.argmax(logits,1), tf.argmax(y,1))
accuracy_op = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
tf.add_to_collection('accuracy_op', accuracy_op)

# Define the batch dataset and run the training:
def evaluate_data(X, Y):
    loss, acc  = sess.run([loss_op, accuracy_op], feed_dict = {x: X, y: Y, keep_prob: 1.0})
    return loss, acc

with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())
    loss_train, acc_train = sess.run([loss_op, accuracy_op], feed_dict = {x:train_x, y:train_y, keep_prob: 1.0})
    loss_valid, acc_valid = sess.run([loss_op, accuracy_op], feed_dict = {x:valid_x, y:valid_y, keep_prob: 1.0})
    loss_test,  acc_test  = sess.run([loss_op, accuracy_op], feed_dict = {x:test_x,  y:test_y,  keep_prob: 1.0})
    print('train loss:{:.4f} validation loss:{:.4f} train accuracy:{:.4f} validation accuracy:{:.4f}'.format(loss_train, loss_valid, acc_train, acc_valid))

##    from visualization import plot_learning_curves
##    plot_learning_curves(train_losses, train_accuracies, dev_losses, dev_accuracies)
